In [1]:
import pandas as panda
import numpy as np

import joblib

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
# 1. Carregar e explorar os dados

df = panda.read_csv("imob_dados.csv")
df.head(10)

,Bairro,TipoImovel,Area,Quartos,Vagas,ValorAluguel
0,Centro,Apartamento,75,2,1,1800
1,Centro,Apartamento,90,3,2,2500
2,Mauricio de Nassau,Apartamento,120,3,2,3200
3,Mauricio de Nassau,Casa,180,4,3,4500
4,Indianopolis,Kitnet,35,1,0,900
5,Indianopolis,Apartamento,70,2,1,1600
6,Universitario,Kitnet,30,1,0,850
7,Universitario,Apartamento,65,2,1,1500
8,Boa Vista,Casa,200,4,2,4200
9,Boa Vista,Apartamento,85,3,1,2200


In [3]:
print("Quantidade de linhas e colunas: ", df.shape)
df.info()
df.isna().sum()

Quantidade de linhas e colunas:  (300, 6)
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Bairro        300 non-null    str  
 1   TipoImovel    300 non-null    str  
 2   Area          300 non-null    int64
 3   Quartos       300 non-null    int64
 4   Vagas         300 non-null    int64
 5   ValorAluguel  300 non-null    int64
dtypes: int64(4), str(2)
memory usage: 19.9 KB


Bairro          0
TipoImovel      0
Area            0
Quartos         0
Vagas           0
ValorAluguel    0
dtype: int64

In [4]:
# 2. Definir as entradas e respostas

coluna_entrada = [
    "Bairro",
    "TipoImovel",
    "Area",
    "Quartos",
    "Vagas"
]

x = df[coluna_entrada]
y = df["ValorAluguel"]

display(x)
display(y)

,Bairro,TipoImovel,Area,Quartos,Vagas
0,Centro,Apartamento,75,2,1
1,Centro,Apartamento,90,3,2
2,Mauricio de Nassau,Apartamento,120,3,2
3,Mauricio de Nassau,Casa,180,4,3
4,Indianopolis,Kitnet,35,1,0
...,...,...,...,...,...
295,Centro,Casa,95,2,1
296,Mauricio de Nassau,Apartamento,104,3,2
297,Indianopolis,Casa,127,3,2
298,Universitario,Casa,189,4,2


0      1800
1      2500
2      3200
3      4500
4       900
       ... 
295    2450
296    3100
297    3600
298    4350
299     850
Name: ValorAluguel, Length: 300, dtype: int64

In [5]:
# 3. Separar treino e teste
x_treino, x_teste, y_treino, y_teste = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows: ", len(x_treino))
print("Test rows: ", len(x_teste))

Training rows:  240
Test rows:  60


In [6]:
# 4. Preparar os dados
categoricas = [
    "Bairro",
    "TipoImovel"
]

numericas = [
    "Area",
    "Quartos",
    "Vagas"
]

processador = ColumnTransformer(
    transformers=[
        ("Categóricas", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("Numéricas", "passthrough", numericas)
    ]
)

In [7]:
# 5. Criar e treinar o modelo

modelo = Pipeline(steps=[
    ("processamento", processador),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

modelo.fit(x_treino, y_treino)
print("Modelo treinado com sucesso... -_-")

Modelo treinado com sucesso... -_-


In [8]:
# 6. Avaliar as previsões

predictions = modelo.predict(x_teste)

mae = mean_absolute_error(y_teste, predictions)
r2 = r2_score(y_teste, predictions)

print(f"MAE: R$ {mae:.2f}")
print(" ")
print(f"R²: {r2:.3f}")

comparing = x_teste.copy()
comparing["Aluguel real"] = y_teste
comparing["Aluguel previsto"] = predictions
comparing["Aluguel absoluto"] = abs(
    comparing["Aluguel real"] - comparing["Aluguel previsto"]
)

comparing.head(7)

MAE: R$ 216.98
 
R²: 0.952


,Bairro,TipoImovel,Area,Quartos,Vagas,Aluguel real,Aluguel previsto,Aluguel absoluto
203,Universitario,Apartamento,66,2,2,2000,2085.666667,85.666667
266,Mauricio de Nassau,Casa,133,3,2,3900,4098.750000,198.750000
152,Indianopolis,Apartamento,79,3,3,2700,2592.616667,107.383333
9,Boa Vista,Apartamento,85,3,1,2200,2663.250000,463.250000
233,Universitario,Casa,171,4,3,4100,4404.750000,304.750000
226,Mauricio de Nassau,Kitnet,33,1,0,1100,1012.250000,87.750000
196,Mauricio de Nassau,Casa,184,4,3,4900,4736.422619,163.577381


In [9]:
# 7. Comparar com um modelo básico

modelo_basico = DummyRegressor(strategy="mean")

modelo_basico.fit(x_treino, y_treino)
basic_prediction = modelo_basico.predict(x_teste)

mae_basico = mean_absolute_error(y_teste, basic_prediction)

print(f"MAE do modelo básico: R$ {mae_basico:.2f}")
print(" ")
print(f"MAE da floresta aleatório: R$ {mae:2f}")

MAE do modelo básico: R$ 1109.30
 
MAE da floresta aleatório: R$ 216.982113


In [10]:
# 8. Prever o aluguel de novos imóveis

novos_imoveis = panda.DataFrame(
    {
        "Bairro": [
            "Centro",
            "Mauricio de Nassau",
            "Boa Vista",
        ],
        "TipoImovel": [
            "Apartamento",
            "Casa",
            "Kitnet",
        ],
        "Area": [
            80,
            150,
            30,
        ],
        "Quartos": [
            2,
            4,
            1,
        ],
        "Vagas": [
          1,
          2,
          0,  
        ],
    }
)

previsao_alugueis = modelo.predict(novos_imoveis)
novos_imoveis["Aluguel Previsto"] = previsao_alugueis

display(novos_imoveis)

,Bairro,TipoImovel,Area,Quartos,Vagas,Aluguel Previsto
0,Centro,Apartamento,80,2,1,2066.750
1,Mauricio de Nassau,Casa,150,4,2,4201.250
2,Boa Vista,Kitnet,30,1,0,963.575


In [11]:
# 9. Salvar o modelo

joblib.dump(modelo, "modelo_aluguel.pkl")
print("Arquivo de modelo_aluguel salvo... -_-")

Arquivo de modelo_aluguel salvo... -_-


### Análise dos resultados

1. Quais variáveis foram tratadas como categóricas e quais foram tratadas como numéricas?
    ```python
    categoricas = [
        "Bairro",
        "TipoImovel"
    ]

    numericas = [
        "Area",
        "Quartos",
        "Vagas"
    ]
    ```


2. O que o MAE obtido representa, em reais, para a imobiliária?
    - MAE representa o mean absolute error, ou o erro absoluto médio. É o desvio médio entre o valor do aluguel previsto pelo modelo e o valor real cobrado pelos imóveis.


3. Qual foi o R² encontrado e como você interpreta esse resultado?
    - O R² foi de 0,952. Isso pode representar que o modelo consegue explicar 95% da variação nos preços dos aluguéis de acordo com os dados processados.


4. O modelo treinado apresentou um MAE menor que o modelo básico? Justifique com os valores.
    - Sim, o modelo básico com DummyRegressor teve um MAE de R$ 1.109,30, e o modelo da floresta aleatória com RandomForestRegressor teve um MAE de R$ 216,98. Isso é explicado pelo Machine Learning, pois o modelo aprende a estimar os preços com base nas características do imóvel ao invés de somente fazer uma média, como aconteceu no modelo básico.


5. Entre os imóveis de teste, qual apresentou o maior erro absoluto?
    - O aluguel de número 9, que apresentou uma diferença de aproximadamente R$ 463,25 entre o aluguel real e o aluguel previsto.


6. Por que os dados de teste precisam ficar separados dos dados de treinamento?
    - Para garantir uma avaliação realista da capacidade do modelo treinado, para que o modelo tenha precisão mais precisão ao processar os dados de teste.


7. Quais outras informações poderiam ajudar a melhorar as previsões?
    - Mais características sobre o ambiente, como mobília, estado do imóvel e etc.